# Compilador Personalizado — Esqueleto Incremental
**Asignatura:** Autómatas y Compiladores  
**Integrantes:**
- Nombre: Ariel Leiva RUT: 21.991.092-4
- Nombre: Franco Bernal RUT: 21.927.845-4
- Nombre: Felipe Romero RUT: 21.996.748-9
- Nombre: Hugo Palomino RUT: 23.443.703-8

---

## Descripción del Lenguaje

> Describe aquí en qué consiste tu lenguaje. ¿Qué temática tiene? ¿Cuál es su propósito o "universo" (videojuego, cocina, fantasía, etc.)? ¿Qué lo hace original?

**Nombre del lenguaje:** F1 (nombre temporal)

**Temática / Propósito:** Formula 1 General

**Ejemplo de cómo se vería un programa en tu lenguaje:**
```
(escribe aquí un pequeño pseudo-código de cómo imaginas que se vería tu lenguaje antes de implementarlo)
```

---

## Arquitectura del Compilador

Este notebook está organizado de forma **incremental**: cada sesión agrega una fase nueva sobre la anterior.

```
Código Fuente (.txt)
       │
       ▼
 ┌─────────────┐
 │  FASE 1     │  Análisis Léxico   → Tokens
 │  Lexer      │  (Esta sesión)
 └──────┬──────┘
        │
        ▼
 ┌─────────────┐
 │  FASE 2     │  Análisis Sintáctico → AST (Árbol)
 │  Parser     │  (Próxima sesión)
 └──────┬──────┘
        │
        ▼
 ┌─────────────┐
 │  FASE 3     │  Análisis Semántico → Validaciones
 │  Semántico  │  (Sesión futura)
 └──────┬──────┘
        │
        ▼
 ┌─────────────┐
 │  FASE 4     │  Generación de Código → Python ejecutable
 │  Código     │  (Sesión futura)
 └─────────────┘
```

| Fase | Descripción | Estado |
|------|-------------|--------|
| 1. Análisis Léxico | Definición de tokens con ANTLR | **Esta sesión** |
| 2. Análisis Sintáctico | Reglas gramaticales y construcción del AST | Próxima sesión |
| 3. Análisis Semántico | Validación de variables, tipos y reglas lógicas | Sesión futura |
| 4. Generación de Código | Traducción a Python ejecutable | Sesión futura |


---
# Fase 1 — Análisis Léxico

## ¿Qué es el Análisis Léxico?

El **análisis léxico** es la primera fase de un compilador. Su trabajo es leer el código fuente carácter por carácter y agrupar esos caracteres en unidades con significado llamadas **tokens**.

Piénsalo así: si el código fuente fuera una oración en español, el Lexer la dividiría en palabras y les asignaría una categoría gramatical.

```
Código fuente:   "si x > 10 entonces imprimir x"
                          │
                          ▼  Lexer
Tokens:          [SI] [ID:"x"] [MAYOR] [ENTERO:"10"] [ENTONCES] [IMPRIMIR] [ID:"x"]
```

Cada token tiene:
- Un **tipo** (categoría): ¿es un número? ¿una palabra clave? ¿un identificador de variable?
- Un **valor** (lexema): el texto original del código fuente que lo generó

## ¿Qué es ANTLR4?

ANTLR4 (*Another Tool for Language Recognition*) es una herramienta que **genera automáticamente** un Lexer y un Parser a partir de una gramática que tú defines en un archivo `.g4`.

En lugar de escribir el código del Lexer a mano (comparar caracteres uno a uno, manejar estados, etc.), tú describes las reglas de tu lenguaje con una sintaxis clara y ANTLR hace el trabajo pesado.

## Archivos que genera ANTLR al compilar tu `.g4`

| Archivo generado | Descripción |
|-----------------|-------------|
| `NombreLexer.py` | El analizador léxico: convierte texto en tokens |
| `NombreParser.py` | El analizador sintáctico: construye el árbol |
| `NombreVisitor.py` | Interfaz Visitor: para recorrer el árbol retornando valores |
| `NombreListener.py` | Interfaz Listener: para reaccionar a eventos del árbol |
| `Nombre.tokens` | Lista de todos los tokens y sus números asignados |

## Regla de oro del Lexer: el orden importa

ANTLR da prioridad a las reglas del Lexer según **dos criterios**, en este orden:

1. **La regla que coincide con el texto más largo gana** (ej: `>=` gana sobre `>`)
2. **Si hay empate en largo, gana la que aparece primero** en el archivo `.g4`

Por esto, las **palabras clave SIEMPRE deben ir antes que `ID`** (el identificador genérico).  
Si `ID` apareciera primero, la palabra clave `si` sería reconocida como un identificador, no como la estructura condicional.


---
## Celda 1 — Definición de la Gramática (`F1Compiler.g4`)

### Estructura de un archivo `.g4`

Un archivo de gramática ANTLR tiene dos grandes secciones:

1. **Reglas del Parser** (nombres en `minúsculas`): describen la *estructura* del lenguaje. Qué combinaciones de tokens forman instrucciones válidas.
2. **Reglas del Lexer** (nombres en `MAYÚSCULAS`): describen los *tokens* individuales. Qué texto reconoce cada categoría.

### Requisitos mínimos del proyecto (checklist)
- [ ] Al menos **3 tipos de datos** distintos (ej: entero, flotante, booleano, cadena)
- [ ] **Estructura condicional** (equivalente a `if/else`)
- [ ] Al menos **2 estructuras repetitivas** (equivalente a `while` y `for`)
- [ ] **Operadores matemáticos** básicos: `+`, `-`, `*`, `/`
- [ ] Al menos **2 operadores lógicos** (`AND`, `OR`, `NOT` o sus equivalentes creativos)
- [ ] **Impresión** de datos en pantalla
- [ ] **4 funciones matemáticas** (ej: raíz cuadrada, coseno, seno, potencia)

---

### Sobre la gramática de ejemplo

La gramática de abajo es un **punto de partida funcional y genérico**, con vocabulario en español neutro.  
**Tu tarea es adaptarla** con el vocabulario temático de tu lenguaje propio.

Ejemplos de adaptaciones creativas:
- En lugar de `inicio` / `fin` → `"erase una vez"` / `"y colorín colorado"` (temática cuentos)
- En lugar de `entero` → `"monedas"` (temática videojuego), `"gramos"` (temática cocina)
- En lugar de `si` / `sino` → `"acaso"` / `"de lo contrario"`, o `"check"` / `"else_bro"`

Lo importante es que la sintaxis sea **original** y **consistente** con la temática elegida.


---
## Celda 2 — Compilar la Gramática con ANTLR

Esta celda transforma tu archivo `.g4` en los archivos Python que el compilador usará en el resto del proyecto.

**Ejecuta esta celda cada vez que modifiques tu gramática.**

### ¿Qué hace el comando?

| Parte del comando | Significado |
|-------------------|-------------|
| `java -jar antlr-4.9.2-complete.jar` | Ejecuta la herramienta ANTLR usando Java |
| `-Dlanguage=Python3` | Genera el código en Python (por defecto genera Java) |
| `-visitor` | Genera la clase Visitor (necesaria en Fase 3 y 4) |
| `-listener` | Genera la clase Listener (alternativa al Visitor para Fase 3) |
| `MiLenguaje.g4` | El archivo de gramática a procesar |

### ¿Por qué el paso 3 (recargar módulos)?

Python carga los módulos en memoria la primera vez que los importas. Si ya cargó una versión anterior del Lexer o del Parser, **no detectará automáticamente que hay una versión nueva**. El paso 3 fuerza a Python a "olvidar" los módulos viejos para que la próxima importación lea los archivos recién generados.


In [ ]:
import importlib, sys

# Cambia este nombre si renombraste tu gramática en la celda anterior.
GRAMMAR_NAME = 'F1Compiler'

# ── Paso 1: Eliminar archivos anteriores ──────────────────────────────────
# Borramos cualquier .py, .tokens o .interp generado en compilaciones previas.
# Evita que Python use versiones desactualizadas del Lexer o Parser.
!rm -f {GRAMMAR_NAME}*.py {GRAMMAR_NAME}.tokens {GRAMMAR_NAME}.interp

# ── Paso 2: Compilar la gramática .g4 ─────────────────────────────────────
# ANTLR lee tu gramática y genera automáticamente los archivos Python.
# Si hay errores en tu .g4, ANTLR los reportará aquí con número de línea.
!java -jar antlr-4.9.2-complete.jar -Dlanguage=Python3 -visitor -listener {GRAMMAR_NAME}.g4

# ── Paso 3: Recargar módulos en memoria ──────────────────────────────────
# Eliminamos del registro de módulos de Python cualquier versión anterior,
# para que la próxima importación lea los archivos recién generados.
modulos_antlr = [
    f'{GRAMMAR_NAME}Lexer',
    f'{GRAMMAR_NAME}Parser',
    f'{GRAMMAR_NAME}Visitor',
    f'{GRAMMAR_NAME}Listener'
]

for mod in modulos_antlr:
    if mod in sys.modules:
        del sys.modules[mod]

# ── Verificar archivos generados ──────────────────────────────────────────
print(f"\nGramática '{GRAMMAR_NAME}' compilada. Archivos Python generados:")
!ls {GRAMMAR_NAME}*.py



Gramática 'MiLenguaje' compilada. Archivos Python generados:
MiLenguajeLexer.py     MiLenguajeParser.py
MiLenguajeListener.py  MiLenguajeVisitor.py


---
## Celda 3 — Importaciones y Herramientas del Lexer

Esta celda importa todo lo necesario y define dos funciones auxiliares para inspeccionar el trabajo del Lexer.

Estas herramientas son especialmente útiles en **esta sesión**: te permiten ver exactamente qué tokens reconoce tu gramática antes de preocuparte por el Parser o el árbol.


In [ ]:
from antlr4 import InputStream, CommonTokenStream

# Importa el Lexer y el Parser generados por ANTLR a partir de tu gramática.
# Si cambiaste el nombre de la gramática, actualiza los nombres aquí también.
from F1CompilerLexer import F1CompilerLexer
from F1CompilerParser import F1CompilerParser


def mostrar_tokens(codigo_fuente):
    """
    Ejecuta SOLO el Lexer sobre el código fuente y muestra en pantalla
    la lista completa de tokens reconocidos con su tipo, texto, línea y columna.

    Qué verificar al revisar la salida:
      - Las palabras clave de tu lenguaje aparecen con su tipo correcto (no como ID)
      - Los números: enteros como NUMERO_ENTERO, decimales como NUMERO_FLOTANTE
      - Las cadenas entre comillas aparecen como CADENA_LITERAL
      - Los comentarios y espacios NO aparecen (tienen '-> skip' en la gramática)
      - Los identificadores de variables aparecen como ID

    Parámetros:
        codigo_fuente (str): el código en tu lenguaje a analizar
    """
    # InputStream convierte el string Python en un flujo de caracteres que el Lexer puede recorrer
    flujo_entrada = InputStream(codigo_fuente)

    # El Lexer lee el flujo y produce un token a la vez según las reglas del .g4
    lexer = MiLenguajeLexer(flujo_entrada)

    # CommonTokenStream gestiona el flujo de tokens; fill() los procesa todos de una vez
    flujo_tokens = CommonTokenStream(lexer)
    flujo_tokens.fill()

    tokens = flujo_tokens.tokens

    print(f"{'Num':<5} {'Tipo':<25} {'Texto':<30} {'Línea':<8} {'Columna'}")
    print("-" * 80)

    for token in tokens:
        tipo_num = token.type

        # El tipo -1 es el token especial EOF (fin de archivo), no tiene nombre en el mapa
        if tipo_num == -1:
            nombre_tipo = "EOF"
        else:
            # symbolicNames mapea número de tipo → nombre legible (ej: 5 → 'ENTERO')
            nombre_tipo = lexer.symbolicNames[tipo_num] if tipo_num < len(lexer.symbolicNames) else str(tipo_num)

        # repr() muestra los caracteres especiales visiblemente (ej: \n en lugar de salto de línea)
        texto = repr(token.text)
        linea = token.line
        columna = token.column

        print(f"{tipo_num:<5} {nombre_tipo:<25} {texto:<30} {linea:<8} {columna}")


def contar_tokens_por_tipo(codigo_fuente):
    """
    Cuenta cuántos tokens de cada tipo aparecen en el código fuente
    y muestra el resultado ordenado de mayor a menor.

    Útil para tener una visión general de la composición léxica de un programa
    y detectar si algún tipo está siendo reconocido más (o menos) de lo esperado.

    Parámetros:
        codigo_fuente (str): el código en tu lenguaje a analizar
    """
    flujo_entrada = InputStream(codigo_fuente)
    lexer = MiLenguajeLexer(flujo_entrada)
    flujo_tokens = CommonTokenStream(lexer)
    flujo_tokens.fill()

    conteo = {}
    for token in flujo_tokens.tokens:
        tipo_num = token.type
        if tipo_num == -1:
            nombre = "EOF"
        else:
            nombre = lexer.symbolicNames[tipo_num] if tipo_num < len(lexer.symbolicNames) else str(tipo_num)
        conteo[nombre] = conteo.get(nombre, 0) + 1

    print("Resumen de tokens encontrados:")
    print(f"  {'Tipo':<25} {'Cantidad'}")
    print("  " + "-" * 35)
    for nombre, cantidad in sorted(conteo.items(), key=lambda x: -x[1]):
        print(f"  {nombre:<25} {cantidad}")
    print(f"\n  Total (incluyendo EOF): {sum(conteo.values())}")

print("Funciones 'mostrar_tokens' y 'contar_tokens_por_tipo' listas para usar.")


---
## Celda 4 — Código de Ejemplo en tu Lenguaje

Escribe aquí programas de prueba usando el vocabulario que definiste en tu gramática.

Este es el código que el Lexer analizará en las celdas siguientes.

> **Consejo:** Empieza siempre con programas cortos y simples. Si el Lexer falla, es mucho más fácil encontrar el problema en 5 líneas que en 50.


In [ ]:
# ── Programa de prueba principal ─────────────────────────────────────────
# Adapta este ejemplo para que use el vocabulario de TU lenguaje.
# Por ahora usa la gramática genérica de ejemplo como base.

programa_ejemplo = """
inicio

    // Declaración de variables de distintos tipos
    entero contador = 0;
    flotante temperatura = 36.6;
    booleano activo = verdadero;

    // Imprimir valores
    imprimir("Hola desde mi lenguaje!");
    imprimir(contador);
    imprimir(temperatura);

    // Estructura condicional
    si (temperatura > 36.0) {
        imprimir("Temperatura elevada");
    } sino {
        imprimir("Temperatura normal");
    }

    // Ciclo mientras
    mientras (contador < 3) {
        imprimir(contador);
        contador = contador + 1;
    }

    /* Comentario de bloque:
       Los comentarios no deben aparecer en la lista de tokens
       porque tienen la directiva -> skip en la gramática */

    // Funciones matemáticas
    flotante resultado = raiz(16);
    imprimir(resultado);

    flotante angulo = 45.0;
    flotante c = coseno(angulo);
    flotante s = seno(angulo);
    flotante p = potencia(2, 8);

fin
"""

print(f"Programa de ejemplo definido ({len(programa_ejemplo)} caracteres).")
print("Ejecuta la siguiente celda para ver sus tokens.")


---
## Celda 5 — Ejecutar el Análisis Léxico

Corremos el Lexer sobre el programa de ejemplo y examinamos los tokens resultantes.

Recuerda qué buscar:
- Cada palabra clave debe tener **su tipo propio**, no `ID`
- Los números decimales deben ser `NUMERO_FLOTANTE`, los enteros `NUMERO_ENTERO`
- Las cadenas entre comillas deben ser `CADENA_LITERAL`
- Los comentarios y espacios en blanco **no deben aparecer**


In [ ]:
print("=" * 80)
print("ANÁLISIS LÉXICO — Lista completa de tokens")
print("=" * 80)
mostrar_tokens(programa_ejemplo)

print()
print("=" * 80)
print("RESUMEN DE TOKENS POR TIPO")
print("=" * 80)
contar_tokens_por_tipo(programa_ejemplo)


---
## Celda 6 — Visualización del Árbol Sintáctico (AST)

El **Árbol de Sintaxis Abstracta (AST)** es la estructura que el Parser construye a partir de los tokens. Representa la jerarquía gramatical del programa: qué sentencias contiene, cómo se componen, qué subexpresiones tienen.

Esta función ya está lista para usarse. La estudiaremos en detalle en la **Fase 2** (próxima sesión), pero puedes ejecutarla ahora para ver el árbol de tu gramática actual.

### ¿Cómo leer el árbol?
- Los **nodos internos** (con hijos) muestran el nombre de una **regla del Parser**: `program`, `sentencia`, `declaracion`, `expresion`, etc.
- Las **hojas** (sin hijos) muestran el texto concreto de un **token**: `"inicio"`, `42`, `"x"`, `">"`, etc.
- Las **aristas** representan la relación padre → hijo: el padre agrupa a sus hijos


In [ ]:
from graphviz import Digraph
from antlr4 import InputStream, CommonTokenStream
from IPython.display import Image, display


def visualizar_arbol(nodo, parser, grafo=None, id_padre=None, contador=[0]):
    """
    Genera recursivamente un grafo visual del árbol sintáctico producido por ANTLR.

    Estrategia: recorrido en profundidad (DFS).
    Para cada nodo: crear su nodo en el grafo, conectarlo al padre, luego procesar los hijos.

    Parámetros:
        nodo      : el nodo actual del árbol ANTLR (puede ser una regla o un token hoja)
        parser    : instancia del Parser; necesaria para convertir índices de regla a nombres legibles
        grafo     : el objeto Digraph de Graphviz donde se construye el grafo
        id_padre  : ID único del nodo padre, para dibujar la arista padre → este nodo
        contador  : lista con un entero compartido entre todas las llamadas recursivas.
                    Usamos una lista (mutable) en lugar de un int (inmutable) para poder
                    modificar el valor desde dentro de la función sin necesidad de 'global'.
    """
    if grafo is None:
        grafo = Digraph()
        contador[0] = 0   # Reiniciar el contador en cada llamada raíz

    # Determinar la etiqueta de este nodo
    if nodo.getChildCount() == 0:
        # Nodo hoja: es un token concreto, mostramos su texto tal como aparece en el código
        etiqueta = nodo.getText()
    else:
        # Nodo interno: es una regla del Parser, mostramos el nombre de esa regla
        etiqueta = parser.ruleNames[nodo.getRuleIndex()]

    if not etiqueta or etiqueta.strip() == '':
        etiqueta = '<vacío>'

    # Crear este nodo en el grafo con un ID único
    id_actual = str(contador[0])
    grafo.node(id_actual, etiqueta)
    contador[0] += 1

    # Conectar con el padre si existe
    if id_padre is not None:
        grafo.edge(id_padre, id_actual)

    # Procesar todos los hijos recursivamente (de izquierda a derecha)
    for i in range(nodo.getChildCount()):
        hijo = nodo.getChild(i)
        visualizar_arbol(hijo, parser, grafo, id_actual, contador)

    return grafo


def analizar_y_visualizar(codigo_fuente, nombre_archivo='ast'):
    """
    Orquesta el análisis léxico + sintáctico y genera la imagen del árbol.

    Flujo completo:
        1. InputStream: convierte el string en flujo de caracteres
        2. Lexer:       tokeniza el flujo → produce tokens
        3. CommonTokenStream: administra el flujo de tokens para el Parser
        4. Parser:      lee los tokens y construye el árbol según la gramática
        5. Graphviz:    dibuja el árbol y lo guarda como imagen PNG

    Parámetros:
        codigo_fuente  (str): el código fuente en tu lenguaje
        nombre_archivo (str): nombre base del archivo PNG que se generará (sin extensión)
    """
    flujo_entrada = InputStream(codigo_fuente)
    lexer  = F1CompilerLexer(flujo_entrada)
    tokens = CommonTokenStream(lexer)
    parser = F1CompilerParser(tokens)

    # Llamar a la regla raíz de la gramática para iniciar el parseo
    arbol = parser.program()

    # Mostrar el árbol en formato texto (útil para depuración rápida)
    print("Árbol en formato texto:")
    print(arbol.toStringTree(recog=parser))
    print()

    # Generar y mostrar la imagen del árbol
    grafo = visualizar_arbol(arbol, parser)
    grafo.render(nombre_archivo, format='png', cleanup=True)

    print(f"Imagen guardada como '{nombre_archivo}.png'")
    display(Image(filename=f'{nombre_archivo}.png'))

print("Funciones 'visualizar_arbol' y 'analizar_y_visualizar' listas.")


In [ ]:
# Ejecutar el análisis sobre el programa de ejemplo
analizar_y_visualizar(programa_ejemplo, nombre_archivo='ast_ejemplo')


---
## Celda 7 — Casos de Prueba del Lexer

Define al menos **3 programas de prueba** en tu lenguaje y analiza sus tokens.

Recuerda incluir:
- Al menos 2 programas **válidos** (uno simple, uno más complejo que use todas las estructuras)
- Al menos 1 programa con un **error léxico** para verificar que el Lexer lo detecta (un carácter inválido, una palabra mal escrita, etc.)

Esto es esencial antes de pasar a la Fase 2: si el Lexer no reconoce bien los tokens, el Parser tampoco funcionará.


In [ ]:
# ── Prueba 1: básica ──────────────────────────────────────────────────────
# Declaración de variables e impresión. La más simple posible.
prueba_1 = """
inicio
    entero x = 10;
    flotante y = 3.14;
    imprimir(x);
    imprimir(y);
fin
"""

# ── Prueba 2: estructuras de control ──────────────────────────────────────
# Incluye condicional y ciclo. Más compleja que la anterior.
prueba_2 = """
inicio
    entero i = 0;
    si (i == 0) {
        imprimir("i es cero");
    } sino {
        imprimir("i no es cero");
    }
    mientras (i < 5) {
        i = i + 1;
    }
    imprimir(i);
fin
"""

# ── Prueba 3: funciones matemáticas y operadores lógicos ─────────────────
prueba_3 = """
inicio
    flotante a = 9.0;
    flotante raiz_a = raiz(a);
    flotante cos_a = coseno(a);
    flotante sen_a = seno(a);
    flotante pot = potencia(2, 10);
    booleano cond = verdadero y falso;
    booleano neg = no verdadero;
    imprimir(raiz_a);
fin
"""

# ── Ejecutar las pruebas ──────────────────────────────────────────────────
pruebas = [
    ("Prueba 1 — Básica",                        prueba_1),
    ("Prueba 2 — Estructuras de Control",         prueba_2),
    ("Prueba 3 — Funciones Matemáticas y Lógica", prueba_3),
]

for nombre, codigo in pruebas:
    print("\n" + "=" * 80)
    print(f"  {nombre}")
    print("=" * 80)
    mostrar_tokens(codigo)


---
# Fase 2 — Análisis Sintáctico *(próxima sesión)*

El análisis sintáctico verifica que la secuencia de tokens forma una estructura válida según las reglas gramaticales del Parser. Si los tokens son las "palabras", el Parser verifica que la "oración" tiene sentido gramaticalmente.

El resultado es el **Árbol de Sintaxis Abstracta (AST)**, que ya pudiste ver en la Celda 6.

En la próxima sesión trabajaremos en:
- Crear un **ErrorListener personalizado** para mostrar mensajes de error claros y legibles
- Analizar el árbol para distintos programas e identificar cada nodo
- Entender la diferencia entre **Visitor** y **Listener** para recorrer el árbol

```python
# Esto se implementará en la próxima sesión (Fase 2):

# from antlr4.error.ErrorListener import ErrorListener

# class MiErrorListener(ErrorListener):
#     """
#     Reemplaza el ErrorListener por defecto de ANTLR (que imprime mensajes crípticos)
#     por uno que muestra errores con formato claro y los acumula en una lista.
#     """
#     def __init__(self):
#         super().__init__()
#         self.errores = []
#
#     def syntaxError(self, recognizer, offendingSymbol, line, column, msg, e):
#         error = f"Error de sintaxis en línea {line}, columna {column}: {msg}"
#         self.errores.append(error)
#         print(error)
```


---
# Fase 3 — Análisis Semántico *(sesión futura)*

El análisis semántico recorre el AST para validar reglas que la gramática **no puede expresar** con solo sintaxis. Por ejemplo: la gramática puede verificar que hay una declaración, pero no puede verificar que la variable no fue declarada dos veces.

### Reglas semánticas a implementar (deben elegir 3)

Documenten aquí las 3 que eligió su grupo:

> **Regla 1:** ___________________________
>
> **Regla 2:** ___________________________
>
> **Regla 3:** ___________________________

Ejemplos posibles: variable usada sin declarar, variable declarada pero nunca usada, variable declarada dos veces, operación matemática sobre un tipo no numérico, función sin instrucción de retorno, nombres de variables que no siguen la convención del lenguaje.

### Visitor vs Listener: ¿cuál usar?

| Criterio | Listener | Visitor |
|----------|----------|---------|
| ¿Cómo funciona? | Recibe eventos (enter/exit) automáticamente | Tú controlas cuándo visitar los hijos |
| ¿Puede retornar valores? | No (usa variables de instancia) | Sí (cada método retorna un valor) |
| ¿Para qué conviene? | Análisis semántico (recorrer todo el árbol) | Generación de código (necesitas el valor de cada nodo) |

```python
# Estructura base para el Analizador Semántico (se completará en sesión futura):

# from MiLenguajeListener import MiLenguajeListener
# from MiLenguajeParser import MiLenguajeParser

# class AnalizadorSemantico(MiLenguajeListener):
#     def __init__(self):
#         # Tabla de símbolos: guarda info de cada variable declarada
#         # Formato: { 'nombre_variable': {'tipo': 'entero', 'linea': 3, 'usado': False} }
#         self.tabla_simbolos = {}
#         self.errores = []
#
#     def enterDeclaracion(self, ctx):
#         # Se ejecuta al ENTRAR a cualquier nodo 'declaracion' del árbol
#         # TODO: registrar la variable en la tabla de símbolos
#         # TODO: Regla 1 — detectar si ya estaba declarada (duplicado)
#         pass
#
#     def enterAsignacion(self, ctx):
#         # TODO: Regla 2 — verificar que la variable fue declarada antes de usarse
#         pass
#
#     def exitProgram(self, ctx):
#         # Se ejecuta al SALIR del nodo raíz: el programa terminó de recorrerse
#         # TODO: Regla 3 — detectar variables declaradas pero nunca usadas
#         pass
```


---
# Fase 4 — Generación de Código *(sesión futura)*

A partir del AST validado semánticamente, generamos código Python ejecutable. Esto "cierra el ciclo": tomamos un programa escrito en nuestro lenguaje inventado y lo convertimos en algo que una computadora real puede ejecutar.

Usaremos el patrón **Visitor**: visitamos cada nodo del árbol y decidimos qué código Python le corresponde.

```python
# Estructura base para el Generador de Código (se completará en sesión futura):

# from MiLenguajeVisitor import MiLenguajeVisitor

# class GeneradorCodigo(MiLenguajeVisitor):
#     def __init__(self):
#         self.lineas_codigo = []    # Acumula las líneas del código Python generado
#         self.nivel_indent = 0      # Controla la indentación del código de salida
#
#     def indentar(self):
#         """Devuelve la cadena de espacios correspondiente al nivel actual."""
#         return "    " * self.nivel_indent
#
#     def visitProgram(self, ctx):
#         # Nodo raíz: genera el encabezado del archivo Python
#         self.lineas_codigo.append("# Código generado automáticamente")
#         self.visitChildren(ctx)
#         return "\n".join(self.lineas_codigo)
#
#     def visitDeclaracion(self, ctx):
#         # TODO: traducir "entero x = 5;" → "x = 5" en Python
#         pass
#
#     def visitPrint_stmt(self, ctx):
#         # TODO: traducir "imprimir(x);" → "print(x)" en Python
#         pass
#
#     def visitSi_stmt(self, ctx):
#         # TODO: traducir "si (...) { } sino { }" → "if ...: ... else: ..." en Python
#         pass
```


---
## Bitácora del Grupo

Usen este espacio para registrar las decisiones de diseño y el progreso de cada sesión. Es útil para recordar qué hicieron, qué problemas encontraron y cómo los resolvieron.

| Sesión | Fecha | Actividad | Decisiones tomadas | Problemas / Soluciones |
|--------|-------|-----------|-------------------|------------------------|
| 1 | | Análisis Léxico | | |
| 2 | | Análisis Sintáctico | | |
| 3 | | Análisis Semántico | | |
| 4 | | Generación de Código | | |
| 5 | | Integración y pruebas finales | | |

---
*Autómatas y Compiladores — Semestre 1, 2026 — Profesores: Marcelo Becerra y Emanuel Vega*
